# 🚨 Anomaly Detection — EDA & Model Comparison
Explore the dataset, visualize anomalies, and compare model performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix

from data.data_loader import load_data, preprocess
from models.isolation_forest import IsolationForestModel
from models.autoencoder import AutoencoderModel

plt.style.use('seaborn-v0_8')
print('✅ Imports done')

## 1. Load & Explore Data

In [ ]:
df = load_data()
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["Class"].mean():.2%}')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Class'].value_counts()
axes[0].bar(['Normal', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Amount distribution by class
df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[1], alpha=0.7, label='Normal', color='steelblue')
df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[1], alpha=0.7, label='Fraud', color='crimson')
axes[1].set_title('Transaction Amount by Class')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature correlations
plt.figure(figsize=(14, 10))
corr = df.drop('Class', axis=1).corr()
sns.heatmap(corr.iloc[:10, :10], annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation (first 10 features)')
plt.tight_layout()
plt.savefig('../data/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Preprocess

In [ ]:
X_train, X_val, X_test, y_test, features = preprocess(df)
print('Preprocessing done!')

## 3. Model Comparison — ROC Curves

In [ ]:
# Load saved models
if_model = IsolationForestModel.load()
ae_model = AutoencoderModel.load()

if_scores = if_model.score(X_test)
ae_scores = ae_model.score(X_test)

# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))

for scores, name, color in [
    (if_scores, 'Isolation Forest', 'steelblue'),
    (ae_scores, 'Autoencoder', 'crimson')
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('../data/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Reconstruction Error (Autoencoder)

In [ ]:
ae_scores_normal = ae_scores[y_test == 0]
ae_scores_fraud  = ae_scores[y_test == 1]

plt.figure(figsize=(10, 5))
plt.hist(ae_scores_normal, bins=100, alpha=0.7, label='Normal', color='steelblue', density=True)
plt.hist(ae_scores_fraud,  bins=50,  alpha=0.7, label='Fraud',  color='crimson',  density=True)
plt.axvline(ae_model.threshold, color='orange', lw=2, linestyle='--', label=f'Threshold = {ae_model.threshold:.4f}')
plt.xlabel('Reconstruction Error')
plt.ylabel('Density')
plt.title('Autoencoder Reconstruction Error Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('../data/reconstruction_error.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Threshold: {ae_model.threshold:.6f}')
print(f'Normal errors  — Mean: {ae_scores_normal.mean():.6f}, Std: {ae_scores_normal.std():.6f}')
print(f'Fraud  errors  — Mean: {ae_scores_fraud.mean():.6f}, Std: {ae_scores_fraud.std():.6f}')